# 01 Data Collection and Observing Preparation

This notebook is the stable entry point for target acquisition and observing preparation.  It summarizes the TNS/Lasair/WISeREP workflow from `README.md` and keeps remote downloads opt-in so the notebook can be opened safely during reporting.

Scientific role: collect candidate metadata, observing windows, finder charts, public spectra, and light curves before the spectral-analysis notebooks interpret the data.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display, Markdown, Image

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "output"
ANALYSIS_DIR = OUTPUT_DIR / "analysis_pipeline"
CONFIG_PATH = PROJECT_ROOT / "configs" / "sn_parameter.json"

## Configuration snapshot

The main observing pipeline uses `configs/sn_parameter.json`.  TNS and Lasair credentials are read from `.env`, which is intentionally not committed.

In [ ]:
config = json.loads(CONFIG_PATH.read_text())
display(config)

## Optional remote-data refresh

Keep `RUN_REMOTE_FETCH = False` for normal report work.  Set it to `True` only when you want to refresh TNS, finder charts, Lasair light curves, and WISeREP spectra.

In [ ]:
RUN_REMOTE_FETCH = False

if RUN_REMOTE_FETCH:
    subprocess.run([sys.executable, "scripts/fetch_target_params.py"], cwd=PROJECT_ROOT, check=True)
    subprocess.run([sys.executable, "scripts/fetch_aux_data.py"], cwd=PROJECT_ROOT, check=True)
else:
    print("Remote refresh skipped. Existing output/ and data/ products are used.")

## Available target products

This inventories the generated observing reports, finder charts, light curves, spectra, and model outputs.

In [ ]:
products = []
for path in sorted(OUTPUT_DIR.glob("*")):
    if not path.is_dir() or path.name == "analysis_pipeline":
        continue
    products.append({
        "target": path.name,
        "reports": len(list(path.glob("sn_report_*.txt"))),
        "finder_charts": len(list(path.glob("finder_*"))),
        "lightcurve_files": len(list((path / "lightcurve").glob("*"))) if (path / "lightcurve").exists() else 0,
        "spectra_files": len(list((path / "spectrum").glob("*"))) if (path / "spectrum").exists() else 0,
        "superfit_files": len(list((path / "superfit").glob("*"))) if (path / "superfit").exists() else 0,
        "tardis_files": len(list((path / "tardis").glob("*"))) if (path / "tardis").exists() else 0,
    })
products_df = pd.DataFrame(products)
display(products_df)

## Target status from analysis pipeline

Run `02_spectral_analysis_pipeline.ipynb` or `scripts/build_analysis_products.py` to update this table.

In [ ]:
status_path = ANALYSIS_DIR / "target_status.csv"
if status_path.exists():
    display(pd.read_csv(status_path))
else:
    print(f"Missing {status_path}. Run the spectral analysis pipeline first.")